In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct
import astropy.io.fits as fits
import tools21cm as t2c
from sklearn.decomposition import FastICA
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib
matplotlib.style.use('/home/ppxjf3/paper_params.mplstyle') 


from astropy.cosmology import Planck13 as cosmoP
from astropy.cosmology import FlatLambdaCDM, LambdaCDM

import astropy.units as u
cosmo = FlatLambdaCDM(H0=71 * u.km / u.s / u.Mpc, Om0=0.27)


In [ ]:
import matplotlib.font_manager
import matplotlib as mpl

column_width=240# call "\the\columnwidth" in LaTeX to find
ppi=72#default ppi, can be left the same

scale=2
fig_width=column_width/ppi*scale#inches
fig_height=3*scale#inches

##SET FONT SIZES
font_small_size = 9
font_medium_size = 12
font_bigger_size =14

plt.rc('font', size=font_small_size) # controls default text sizes
plt.rc('axes', titlesize=font_small_size) # fontsize of the axes title
plt.rc('axes', labelsize=font_medium_size) # fontsize of the x and y labels
plt.rc('xtick', labelsize=font_bigger_size) # fontsize of the tick labels
plt.rc('ytick', labelsize=font_bigger_size) # fontsize of the tick labels
plt.rc('legend', fontsize=font_small_size) # legend fontsize
plt.rc('figure', titlesize=font_bigger_size)

#DPI of MNRAS is 300
mpl.rcParams['figure.dpi'] = 300/scale

In [ ]:
def do_fastica(data, comps):
    shape = data.shape
    f_ica = FastICA(n_components=comps)
    #generate the 4 componets
    S = f_ica.fit_transform(data.reshape((shape[0]*shape[1],shape[2])))
    
    #get mixing matrix
    A = f_ica.mixing_
    
    #make model
    model_fICA = (np.matmul(A,S.T).T).reshape((shape[0],shape[1],shape[2]))
    
    #get resids
    resids_fICA = data - model_fICA #residuals 
    
    return model_fICA, resids_fICA

def pearson_correl(x,y):
 return (np.sum((x-np.mean(x))*(y-np.mean(y))))/(np.sqrt(np.sum((x-np.mean(x))**2)*np.sum((y-np.mean(y))**2)))

def zFromNu(nu):
    """
    Convert frequency of 21cm line to redshift
    
    Input: nu [MHz]
    """
    nu21 = 1.420405e3  #MHz
    return nu21/nu - 1.0

def NuFromz(z):
    """
    Convert frequency of 21cm line to redshift
    
    Input: nu [MHz]
    """
    nu21 = 1.420405e3  #MHz
    return nu21/(z + 1.0)

def correlation_graph(LC, RSD_LC, res_LC, res_RSD, freq, name):
    cc_LC = np.empty([res_LC.shape[2]])
    cc_RSD = np.empty([res_RSD.shape[2]])

    for ii in range(0,res_RSD.shape[2]):
        cc_LC[ii] = pearson_correl(res_LC[:,:,ii],LC[:,:,ii])
        cc_RSD[ii] = pearson_correl(res_RSD[:,:,ii],RSD_LC[:,:,ii])
    
    plt.plot(freq, cc_RSD, label = 'RSD')
    plt.plot(freq, cc_LC, label = 'LC')
    plt.ylabel('correlation')
    plt.xlabel('freq')
    plt.legend()
    plt.savefig('/home/ppxjf3/RSD_LC/comparison/graphs/' + name, dpi=330)
    plt.show()
    
    plt.plot(freq, cc_RSD - cc_LC)
    plt.ylabel('residual (cc_RSD - cc_LC)')
    plt.xlabel('freq')
    plt.show()

def get_lengths(nu_low, nu_hi, nu_mid, theta_FOV):
    z_lo = zFromNu(nu_low)
    z_mid = zFromNu(nu_mid)
    z_hi = zFromNu(nu_hi)
    
    L_para = cosmo.comoving_distance(z_lo) - cosmo.comoving_distance(z_hi)
    L_perp = cosmo.comoving_distance(z_mid) * (np.pi * theta_FOV / 180.0)
    return L_para/u.Mpc, L_perp/u.Mpc

def LC_Volume(nu_min, nu_max, FoV):
    z_lo = zFromNu(nu_min)
    z_hi = zFromNu(nu_max)
    
    r_min = cosmo.comoving_distance(z_lo)/u.Mpc
    r_max = cosmo.comoving_distance(z_hi)/u.Mpc
    height = np.abs(r_max - r_min)
    
    S1 = (r_min* (np.pi * FoV / 180.0))**2
    S2 = (r_max* (np.pi * FoV / 180.0))**2
    return (height/3)*(S1 + S2 + np.sqrt(S1*S2))

In [ ]:
base = '/home/ppxjf3/RSD_LC/comparison/FINAL_LIGHTCONES_PEC_VEL_PAPER/'
npix=512
nfreq=201
#with open(base + 'LightconeRSD_N512_FOV1.0000_dnu0.10MHz_095.00MHz_075.00MHz_ds0.002976_div00.00_pv1_oneevent0_evo1_lcon0_dz_000.10.dat', "rb") as fid:
with open(base+'LightconeRSD_N512_FOV1.0000_dnu0.10MHz_095.00MHz_075.00MHz_ds0.003099_div00.00_pv1_oneevent0_evo1_lcon0_dz_000.10.dat', "rb") as fid:
    # Read the binary data from the file
    data = fid.read()

    # Unpack the binary data into a tuple of floats
    RSD = struct.unpack('f' * (len(data) // struct.calcsize('f')), data)

RSD_z01_nu01_1deg = np.array(RSD)*1e3
RSD_z01_nu01_1deg = RSD_z01_nu01_1deg.reshape(npix,npix,nfreq)
        
with open(base +"LightconeRSD_N512_FOV1.0000_dnu0.10MHz_095.00MHz_075.00MHz_ds0.007747_div00.00_pv1_oneevent0_evo1_lcon1_dz_000.10.dat", "rb") as fid:
    # Read the binary data from the file
    data = fid.read()

    # Unpack the binary data into a tuple of floats
    LC = struct.unpack('f' * (len(data) // struct.calcsize('f')), data)

LC_z01_nu01_1deg = np.array(LC)*1e3
LC_z01_nu01_1deg = LC_z01_nu01_1deg.reshape(npix,npix,nfreq)

In [ ]:
i0 = 50
i1 = 151

BINS = np.arange(np.min([LC_z01_nu01_1deg[:,:,i0:i1], RSD_z01_nu01_1deg[:,:,i0:i1]]), np.max([LC_z01_nu01_1deg[:,:,i0:i1], RSD_z01_nu01_1deg[:,:,i0:i1]]),10.0)

fig, ax = plt.subplots(figsize=(fig_width, fig_height),
                       nrows=1, ncols=1,)
ax.hist(RSD_z01_nu01_1deg[:,:,i0:i1].reshape((512*512*101)), bins=BINS, color='blue', alpha=0.5, label='extended')
ax.hist(LC_z01_nu01_1deg[:,:,i0:i1].reshape((512*512*101)), bins=BINS, color='orange', alpha=0.5, label='basic')
ax.set_yscale('log')
ax.set_ylabel(r'$N_{\rm pixels}$', fontsize=16)
ax.set_xlabel(r'$\delta T_{\rm b} /\rm mK$', fontsize=16)
plt.legend(frameon=False, fontsize=14)
plt.tight_layout()
plt.savefig(base + 'graphs_for_paper/CD_deltaTb_hist.pdf', dpi=330)
plt.show()

print(np.max(LC_z01_nu01_1deg[:,:,i0:i1]))
print(np.max(RSD_z01_nu01_1deg[:,:,i0:i1]))
print(np.min(LC_z01_nu01_1deg[:,:,i0:i1]))
print(np.min(RSD_z01_nu01_1deg[:,:,i0:i1]))

In [ ]:

with fits.open(base + 'Fg_N512_FOV1.0_75_95.0MHz_0.1MHz.fits', memmap=True) as hdu:
    FG = np.array(hdu[0].data)  
    
FG=FG*1e3
fig, ax = plt.subplots(1,1)
pmc_lc_cd = ax.imshow(FG[:,:,20])
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(pmc_lc_cd, cax=cax, orientation='vertical', label='mK')

FG_LC = FG + LC_z01_nu01_1deg
FG_RSD = FG + RSD_z01_nu01_1deg

freq01 = np.arange(75, 95.1, 0.1)

In [ ]:
c = 4
model1, resids_LC_FG = do_fastica(FG_LC, c)
model2, resids_RSD_FG = do_fastica(FG_RSD, c)


In [ ]:
from scipy import interpolate
import matplotlib.colors as colors

plotting_scale={'x': 'log', 'y': 'log', 'z': 'log'}
bins = 8

i0 = 50
i1 = 151

L_para, L_perp = get_lengths(freq01[i0], freq01[i1], (freq01[i0]+freq01[i1])/2, 1.0)
box_dims = [L_perp, L_perp, L_para]

pp_RSD, kper_RSD, kpar_RSD, n_RSD = t2c.power_spectrum_2d(RSD_z01_nu01_1deg[:,:,i0:i1], kbins=bins, box_dims= box_dims, return_modes=True)
for ii in range(0,len(kper_RSD)):
    for jj in range(0,len(kpar_RSD)):
        pp_RSD[ii,jj] = (pp_RSD[ii,jj]*np.sqrt(kper_RSD[ii]**2 + kpar_RSD[jj]**2)**3)/(2*np.pi**2)
fp_RSD = interpolate.interp2d(kper_RSD, kpar_RSD, pp_RSD.T, kind='linear')
X_RSD, Y_RSD = kper_RSD, kpar_RSD
C_RSD = fp_RSD(X_RSD,Y_RSD)
norm_RSD = colors.LogNorm(vmin=C_RSD[np.isfinite(C_RSD)].min(), vmax=C_RSD[np.isfinite(C_RSD)].max()) if plotting_scale['z']=='log' else None 

pp_RSD_RES, kper_RSD_RES, kpar_RSD_RES, n_RSD_RES = t2c.power_spectrum_2d(resids_RSD_FG[:,:,i0:i1], kbins=bins, box_dims= box_dims, return_modes=True)
for ii in range(0,len(kper_RSD_RES)):
    for jj in range(0,len(kpar_RSD_RES)):
        pp_RSD_RES[ii,jj] = (pp_RSD_RES[ii,jj]*np.sqrt(kper_RSD_RES[ii]**2 + kpar_RSD_RES[jj]**2)**3)/(2*np.pi**2)
err_RSD_RES = 1/np.sqrt(n_RSD_RES)
fp_RSD_RES = interpolate.interp2d(kper_RSD_RES, kpar_RSD_RES, pp_RSD_RES.T, kind='linear')
X_RSD_RES, Y_RSD_RES = kper_RSD_RES, kpar_RSD_RES
C_RSD_RES = fp_RSD_RES(X_RSD_RES,Y_RSD_RES)
norm_RSD_RES = colors.LogNorm(vmin=C_RSD_RES[np.isfinite(C_RSD_RES)].min(), vmax=C_RSD_RES[np.isfinite(C_RSD_RES)].max()) if plotting_scale['z']=='log' else None 


pp_LC, kper_LC, kpar_LC, n_LC = t2c.power_spectrum_2d(LC_z01_nu01_1deg[:,:,i0:i1], kbins=bins, box_dims= box_dims, return_modes=True)
for ii in range(0,len(kper_LC)):
    for jj in range(0,len(kpar_LC)):
        pp_LC[ii,jj] = (pp_LC[ii,jj]*np.sqrt(kper_LC[ii]**2 + kpar_LC[jj]**2)**3)/(2*np.pi**2)
fp_LC = interpolate.interp2d(kper_LC, kpar_LC, pp_LC.T, kind='linear')
X_LC, Y_LC = kper_LC, kpar_LC
C_LC = fp_LC(X_LC,Y_LC)
norm_LC = colors.LogNorm(vmin=C_LC[np.isfinite(C_LC)].min(), vmax=C_LC[np.isfinite(C_LC)].max()) if plotting_scale['z']=='log' else None 


pp_LC_RES, kper_LC_RES, kpar_LC_RES, n_LC_RES = t2c.power_spectrum_2d(resids_LC_FG[:,:,i0:i1], kbins=bins, box_dims= box_dims, return_modes=True)
for ii in range(0,len(kper_LC_RES)):
    for jj in range(0,len(kpar_LC_RES)):
        pp_LC_RES[ii,jj] = (pp_LC_RES[ii,jj]*np.sqrt(kper_LC_RES[ii]**2 + kpar_LC_RES[jj]**2)**3)/(2*np.pi**2)
err_LC_RES = 1/np.sqrt(n_LC_RES)
fp_LC_RES = interpolate.interp2d(kper_LC_RES, kpar_LC_RES, pp_LC_RES.T, kind='linear')
X_LC_RES, Y_LC_RES = kper_LC_RES, kpar_LC_RES
C_LC_RES = fp_LC_RES(X_LC_RES,Y_LC_RES)
norm_LC_RES = colors.LogNorm(vmin=C_LC_RES[np.isfinite(C_LC_RES)].min(), vmax=C_LC_RES[np.isfinite(C_LC_RES)].max()) if plotting_scale['z']=='log' else None 

C_tot = np.concatenate((C_LC, C_RSD, C_LC_RES, C_RSD_RES))

norm_tot = colors.LogNorm(vmin=C_tot[np.isfinite(C_tot)].min(), vmax=C_tot[np.isfinite(C_tot)].max()) if plotting_scale['z']=='log' else None 

C_diff =(np.abs(C_RSD - C_LC)/C_LC)*100

norm_diff = colors.LogNorm(vmin=C_diff[np.isfinite(C_diff)].min(), vmax=C_diff[np.isfinite(C_diff)].max()) if plotting_scale['z']=='log' else None 


C_diff_res =  (np.abs(C_RSD_RES - C_LC_RES)/C_LC_RES)*100
norm_diff_res = colors.LogNorm(vmin=C_diff_res[np.isfinite(C_diff_res)].min(), vmax=C_diff_res[np.isfinite(C_diff_res)].max()) if plotting_scale['z']=='log' else None 

C_tot_diff = np.concatenate((C_diff, C_diff_res))
norm_tot_diff = colors.LogNorm(vmin=C_tot_diff[np.isfinite(C_tot_diff)].min(), vmax=C_tot_diff[np.isfinite(C_tot_diff)].max()) if plotting_scale['z']=='log' else None 


In [ ]:

import matplotlib.font_manager
import matplotlib as mpl

k_mag =  0.3783
theta = np.linspace(0, 2*np.pi, 500)
k_par = k_mag * np.cos(theta)
k_perp = k_mag * np.sin(theta)
mask = (k_par > 0) & (k_perp > 0)

fig, ax = plt.subplots(figsize=(2*fig_width, fig_height),
                       nrows=2, ncols=3,)
    

pcm_RSD = ax[0,0].pcolormesh(X_RSD, Y_RSD, C_RSD, norm=norm_tot, cmap='viridis')
cbar= plt.colorbar(pcm_RSD, ax=ax[0,0], pad=0.01)
cbar.set_label(r'$\Delta^2_{\rm 3D}(k_{\parallel}, k_{\perp}) /\rm mK^2 $', size=16) 
ax[0,0].set_xlabel(r'$k_{\perp}/ \rm Mpc^{-1}$', fontsize = 14)
ax[0,0].set_ylabel(r'$k_{\parallel} /\rm Mpc^{-1}$', fontsize = 14)
ax[0,0].set_xscale(plotting_scale['x'])
ax[0,0].set_yscale(plotting_scale['y'])
ax[0,0].plot(k_perp[mask], k_par[mask], color='k', lw=1.5, linestyle='--', label=r'$|\vec{k}| = 40$')
ax[0,0].set_ylim(0.025,1.25)
ax[0,0].set_xlim(left=0.028)

pcm_LC = ax[0,1].pcolormesh(X_LC, Y_LC, C_LC, norm=norm_tot, cmap='viridis')
cbar = plt.colorbar(pcm_LC, ax=ax[0,1], pad=0.01)
cbar.set_label(r'$\Delta^2_{\rm 3D}(k_{\parallel}, k_{\perp}) /\rm mK^2 $', size=16) 
ax[0,1].set_xlabel(r'$k_{\perp}/ \rm Mpc^{-1}$', fontsize = 14)
ax[0,1].set_ylabel(r'$k_{\parallel} /\rm Mpc^{-1}$', fontsize = 14)
ax[0,1].set_xscale(plotting_scale['x'])
ax[0,1].set_yscale(plotting_scale['y'])
ax[0,1].plot(k_perp[mask], k_par[mask], color='k', lw=1.5, linestyle='--', label=r'$|\vec{k}| = 40$')
ax[0,1].set_ylim(0.025,1.25)
ax[0,1].set_xlim(left=0.028)

pcm_LC = ax[0,2].pcolormesh(X_LC, Y_LC, C_diff, norm=norm_tot_diff, cmap='viridis')
cbar= plt.colorbar(pcm_LC, ax=ax[0,2], pad=0.01)
cbar.set_label(r'$[|\Delta_{\rm ext.}^2 - \Delta^2_{\rm basic}|/\Delta^2_{\rm basic}] \times 100 $', size=14) 
ax[0,2].set_ylabel(r'$k_{\parallel} /\rm Mpc^{-1}$', fontsize = 14)
ax[0,2].set_xscale(plotting_scale['x'])
ax[0,2].set_yscale(plotting_scale['y'])
ax[0,2].plot(k_perp[mask], k_par[mask], color='k', lw=1.5, linestyle='--', label=r'$|\vec{k}| = 40$')
ax[0,2].set_ylim(0.025,1.25)
ax[0,2].set_xlim(left=0.028)    

pcm_RSD_RES = ax[1,0].pcolormesh(X_RSD_RES, Y_RSD_RES, C_RSD_RES, norm=norm_tot, cmap='viridis')
cbar= plt.colorbar(pcm_RSD_RES, ax=ax[1,0], pad=0.01)
cbar.set_label(r'$\Delta^2_{\rm 3D}(k_{\parallel}, k_{\perp}) /\rm mK^2 $', size=16) 
ax[1,0].set_xlabel(r'$k_{\perp}/ \rm Mpc^{-1}$', fontsize = 14)
ax[1,0].set_ylabel(r'$k_{\parallel} /\rm Mpc^{-1}$', fontsize = 14)
ax[1,0].set_xscale(plotting_scale['x'])
ax[1,0].set_yscale(plotting_scale['y'])
ax[1,0].plot(k_perp[mask], k_par[mask], color='k', lw=1.5, linestyle='--', label=r'$|\vec{k}| = 40$')
ax[1,0].set_ylim(0.025,1.25)
ax[1,0].set_xlim(left=0.028)

pcm_LC_RES = ax[1,1].pcolormesh(X_LC_RES, Y_LC_RES, C_LC_RES, norm=norm_tot, cmap='viridis')
cbar= plt.colorbar(pcm_LC_RES, ax=ax[1,1], pad=0.01)
cbar.set_label(r'$\Delta^2_{\rm 3D}(k_{\parallel}, k_{\perp}) /\rm mK^2 $', size=16) 
ax[1,1].set_xlabel(r'$k_{\perp}/ \rm Mpc^{-1}$', fontsize = 14)
ax[1,1].set_ylabel(r'$k_{\parallel} /\rm Mpc^{-1}$', fontsize = 14)
ax[1,1].set_xscale(plotting_scale['x'])
ax[1,1].set_yscale(plotting_scale['y'])
ax[1,1].plot(k_perp[mask], k_par[mask], color='k', lw=1.5, linestyle='--', label=r'$|\vec{k}| = 40$')
ax[1,1].set_ylim(0.025,1.25)
ax[1,1].set_xlim(left=0.028)  

pcm_LC_RES = ax[1,2].pcolormesh(X_LC_RES, Y_LC_RES, C_diff_res, norm=norm_tot_diff, cmap='viridis')
cbar= plt.colorbar(pcm_LC_RES, ax=ax[1,2], pad=0.01)
cbar.set_label(r'$[|\Delta_{\rm ext.}^2 - \Delta^2_{\rm basic}|/\Delta^2_{\rm basic}] \times 100 $', size=14) 
ax[1,2].set_xlabel(r'$k_{\perp}/ \rm Mpc^{-1}$', fontsize = 14)
ax[1,2].set_ylabel(r'$k_{\parallel} /\rm Mpc^{-1}$', fontsize = 14)
ax[1,2].set_xscale(plotting_scale['x'])
ax[1,2].set_yscale(plotting_scale['y'])
ax[1,2].plot(k_perp[mask], k_par[mask], color='k', lw=1.5, linestyle='--', label=r'$|\vec{k}| = 40$')
ax[1,2].set_ylim(0.025,1.25)
ax[1,2].set_xlim(left=0.028)
 
plt.tight_layout()
plt.savefig(base + 'graphs_for_paper/cyclindrical_powerSpec_FGremoval_512_80-90MHz.pdf', dpi=330, bbox_inches='tight')
plt.show()

In [ ]:
kbins=12
mubins=15
    
Pk_lc, mubins_lc, kbins_lc, nmode_lc = t2c.power_spectrum_mu(LC_z01_nu01_1deg[:,:,i0:i1], los_axis = 2, box_dims=box_dims, mubins=mubins,kbins=kbins, exclude_zero_modes=True,return_n_modes=True,absolute_mus=False)
Pk_reslc, mubins_reslc, kbins_reslc, nmode_reslc = t2c.power_spectrum_mu(resids_LC_FG[:,:,i0:i1], los_axis = 2, box_dims=box_dims, mubins=mubins,kbins=kbins, exclude_zero_modes=True,return_n_modes=True,absolute_mus=False)

Pk_rsd, mubins_rsd, kbins_rsd, nmode_rsd = t2c.power_spectrum_mu(RSD_z01_nu01_1deg[:,:,i0:i1], los_axis = 2, box_dims=box_dims, mubins=mubins,kbins=kbins, exclude_zero_modes=True,return_n_modes=True,absolute_mus=False)
Pk_resrsd, mubins_resrsd, kbins_resrsd, nmode_resrsd = t2c.power_spectrum_mu(resids_RSD_FG[:,:,i0:i1], box_dims=box_dims, los_axis = 2, mubins=mubins,kbins=kbins, exclude_zero_modes=True,return_n_modes=True,absolute_mus=False)

dk_lc = (Pk_lc*kbins_lc**3)/(2*np.pi**2)
dk_rsd = (Pk_rsd*kbins_rsd**3)/(2*np.pi**2)
dk_lc_res = (Pk_reslc*kbins_reslc**3)/(2*np.pi**2)
dk_rsd_res = (Pk_resrsd*kbins_resrsd**3)/(2*np.pi**2)


In [ ]:
#calculate errors
err_lc_res= np.empty_like(nmode_reslc)
err_rsd_res= np.empty_like(nmode_resrsd)

for ii in range(0,nmode_lc.shape[0]):
    for jj in range(0,nmode_lc.shape[1]):
        err_lc_res[ii, jj] = dk_lc_res[ii, jj]/np.sqrt(nmode_reslc[ii, jj])
        
        err_rsd_res[ii, jj] = dk_rsd_res[ii, jj]/np.sqrt(nmode_resrsd[ii,jj])

In [ ]:
ii = 2
jj = 4

import matplotlib.ticker as ticker

def format_ticks(value, pos):
    """Format ticks to show integers while maintaining the logarithmic scale."""
    exponent = int(np.log10(value))
    mantissa = value / (10 ** exponent)
    if np.isclose(mantissa, 1.0):  # Only show ticks at powers of ten
        return f'$10^{{{exponent}}}$'
    return ''

fig, ax = plt.subplots(figsize=(fig_width, fig_height), nrows=2, ncols=1)

# Upper panel
ax[0].plot(mubins_lc, dk_lc[:, ii], color='k', label=r'$\delta T_b(21 \rm cm)$')
ax[0].errorbar(mubins_lc, dk_lc_res[:, ii], err_lc_res[:, ii], color='k', ls='none', capsize=2.5, label=r'$\delta T_b(\rm reconstructed)$', marker='x')
ax[0].plot(mubins_rsd, dk_rsd[:, ii], color='b')
ax[0].errorbar(mubins_resrsd, dk_rsd_res[:, ii], err_rsd_res[:, ii], color='b', ls='none', capsize=2.5, marker='x')
#freq_ticks = [0.0001, 0.0002]

# Set the tick locations on the secondary x-axis
#ax[0].set_xticks(freq_ticks)  # Set the tick positions in MHz
ax[0].set_yscale('log')
ax[0].set_xlabel(r'$\mu$', fontsize=16)
ax[0].set_ylabel(r'$\Delta^2_{\rm 3D}(k,\mu)/ \rm mK^2$', fontsize=16)
ax[0].text(0.95, 0.15, r' $k={0:.2f}$'.format(kbins_lc[ii]), transform=ax[0].transAxes, ha='right', va='top', fontsize=14)
ax[0].legend(frameon=False, fontsize=14)


# Lower panel
ax[1].plot(kbins_lc, dk_lc[jj, :], color='k')
ax[1].errorbar(kbins_reslc, dk_lc_res[jj, :], err_lc_res[jj, :], color='k', ls='none', capsize=2.5, marker='x')
ax[1].plot(kbins_rsd, dk_rsd[jj, :], color='b')
ax[1].errorbar(kbins_resrsd, dk_rsd_res[jj, :], err_rsd_res[jj, :], color='b', ls='none', capsize=2.5, marker='x')
ax[1].set_xlabel(r'$k/(\rm Mpc)^{-1}$', fontsize=16)
ax[1].set_ylabel(r'$\Delta^2_{\rm 3D}(k,\mu)/ \rm mK^2$', fontsize=16)
ax[1].text(0.95, 0.15, r' $\mu={0:.2f}$'.format(mubins_lc[jj]), transform=ax[1].transAxes, ha='right', va='top', fontsize=14)
ax[1].set_yscale('log')
ax[1].set_xscale('log')
ax[1].axvline(0.3783, linestyle='dashed', color='k', alpha=0.5)
ax[1].set_xlim(left = 0.07)
ax[1].legend(frameon=False, fontsize=14)

plt.tight_layout()
plt.savefig(base + 'graphs_for_paper/mu_ps_fg_removal_cd.pdf', dpi=330)
plt.show()
